# Tests

In [1]:
# Import packages.
import demes
import demesdraw
from itertools import combinations
import matplotlib
from matplotlib import pyplot as plt
import msprime
import numpy as np
import pandas as pd
import stdpopsim
import tskit

# Initialize my pandas preferences.
pd.set_option("display.max_columns", None)

# Print the version.
print("demes", demes.__version__)
print("demesdraw", demesdraw.__version__)
print("matplotlib", matplotlib.__version__)
print("msprime", msprime.__version__)
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("stdpopsim", stdpopsim.__version__)
print("tskit", tskit.__version__)

demes 0.2.3
demesdraw 0.4.0
matplotlib 3.10.6
msprime 1.3.4
numpy 2.3.3
pandas 2.3.3
stdpopsim 0.3.0
tskit 0.6.4


## Q: Are the iterations and length columns being recorded?
See `tests/README.md` for the commands.

In [ ]:
# Load and display the outputs.
Demography1_columns_test_precommit_df = pd.read_csv(
    "Demography1_columns_test_precommit.txt", sep="\t"
)
PapuansOutOfAfrica_10J19_columns_test_precommit_df = pd.read_csv(
    "PapuansOutOfAfrica_10J19_columns_test_precommit.txt", sep="\t"
)

print("#***# Demography1 #***#")
display(Demography1_columns_test_precommit_df.head())

print("#***# PapuansOutOfAfrica_10J19 #***#")
display(PapuansOutOfAfrica_10J19_columns_test_precommit_df.head())

#***# Demography1 #***#


,iteration,haplotype,pop,start,end,length,admix_event,admixtime,snps,Seq_DEN
0,0,Ingroup,126443,158290,Distant_DEN>Ingroup,1551.7,7,5,NaN,NaN
1,0,Ingroup,206139,223580,Distant_DEN>Ingroup,1551.7,8,4,NaN,NaN
2,0,Ingroup,1006542,1007305,Distant_DEN>Ingroup,1551.7,0,0,NaN,NaN
3,0,Ingroup,1180240,1290919,Distant_DEN>Ingroup,1551.7,27,13,NaN,NaN
4,0,Ingroup,1342407,1344305,Distant_DEN>Ingroup,1551.7,2,0,NaN,NaN


#***# PapuansOutOfAfrica_10J19 #***#


,iteration,haplotype,pop,start,end,length,admix_event,admixtime,snps,DenA,NeaA
0,0,Papuan,244738,253672,CHB>Papuan,1656.3,1,0,0,NaN,NaN
1,0,Papuan,359691,459872,CHB>Papuan,1656.3,32,0,0,NaN,NaN
2,0,Papuan,561083,572585,CHB>Papuan,1656.3,2,0,0,NaN,NaN
3,0,Papuan,845473,918344,CHB>Papuan,1409.3,19,0,0,NaN,NaN
4,0,Papuan,1526912,1706283,CHB>Papuan,386.5,46,0,8,NaN,NaN


In [ ]:
# Sanity check the number of iterations.
# Expected output: [0, 1, 2, 3, 4]
print(
    f"Demography1 iterations:{Demography1_columns_test_precommit_df.iteration.unique()}"
)
print(
    f"PapuansOutOfAfrica_10J19 iterations:{PapuansOutOfAfrica_10J19_columns_test_precommit_df.iteration.unique()}"
)

Demography1 iterations:[0 1]
PapuansOutOfAfrica_10J19 iterations:[ 0  1  2  3  4  5 10 11]


In [ ]:
# Sanity check the segment lengths.
# Note that based on the cells above the end = start and start = pop.
Demography1_columns_test_precommit_df["end_minus_start"] = (
    Demography1_columns_test_precommit_df["start"]
    - Demography1_columns_test_precommit_df["pop"]
)
PapuansOutOfAfrica_10J19_columns_test_precommit_df["end_minus_start"] = (
    PapuansOutOfAfrica_10J19_columns_test_precommit_df["start"]
    - PapuansOutOfAfrica_10J19_columns_test_precommit_df["pop"]
)


print(
    f"Demography1 does end - start = length?: {Demography1_columns_test_precommit_df['end_minus_start'].equals(Demography1_columns_test_precommit_df['length'])}"
)
print(
    f"PapuansOutOfAfrica_10J19 does end - start = length?: {PapuansOutOfAfrica_10J19_columns_test_precommit_df['end_minus_start'].equals(PapuansOutOfAfrica_10J19_columns_test_precommit_df['length'])}"
)

Demography1 does end - start = length?: False
PapuansOutOfAfrica_10J19 does end - start = length?: False


A: No, neither columns are being recorded.
Based on commit `104fa43`, the culprit seems to be lines 268-286, where it is clear to see that neither are being recorded:
```python
        for (haplotype, pop, start, end,  admix_event, admixtime) in introgressed_segments:
            

            OUTPUT = [haplotype, pop, start, end,  admix_event, admixtime]
            ID = f'{haplotype}|{pop}|{admix_event}|{admixtime}|{start}|{end}'
            
            snps = introgressed_segments_dict[ID, 'snps']
            shared = [str(introgressed_segments_dict[ID, seq_arch]) for seq_arch in ARCHAIC_GENOME_SAMPLES]
            OUTPUT += [snps, '\t'.join(shared)]

            if args.extrainfo:
                snp_ages = ','.join([x for x in introgressed_snp_ages[ID, 'snp_ages']])
                snp_position = ','.join([x for x in introgressed_snp_ages[ID, 'snp_positions']])
                matches = ','.join([x for x in introgressed_snp_ages[ID, 'match']])

                OUTPUT += [snp_position, snp_ages, matches]

            if any([keyword in admix_event for keyword in keywords]):
                print(*OUTPUT, sep = '\t', file = out)
```
Luckily, Skov writes nice code so the fix is easy:
```python
        for (haplotype, pop, start, end,  admix_event, admixtime) in introgressed_segments:
            

            OUTPUT = [iteration, haplotype, pop, start, end, end - start, admix_event, admixtime]
            ID = f'{haplotype}|{pop}|{admix_event}|{admixtime}|{start}|{end}'
            
            snps = introgressed_segments_dict[ID, 'snps']
            shared = [str(introgressed_segments_dict[ID, seq_arch]) for seq_arch in ARCHAIC_GENOME_SAMPLES]
            OUTPUT += [snps, '\t'.join(shared)]

            if args.extrainfo:
                snp_ages = ','.join([x for x in introgressed_snp_ages[ID, 'snp_ages']])
                snp_position = ','.join([x for x in introgressed_snp_ages[ID, 'snp_positions']])
                matches = ','.join([x for x in introgressed_snp_ages[ID, 'match']])

                OUTPUT += [snp_position, snp_ages, matches]

            if any([keyword in admix_event for keyword in keywords]):
                print(*OUTPUT, sep = '\t', file = out)
```

Now let's see if the iterations and length columns are being recorded in the postcommit files.

In [ ]:
# Load and display the outputs.
Demography1_columns_test_postcommit_df = pd.read_csv(
    "Demography1_columns_test_postcommit.txt", sep="\t"
)
PapuansOutOfAfrica_10J19_columns_test_postcommit_df = pd.read_csv(
    "PapuansOutOfAfrica_10J19_columns_test_postcommit.txt", sep="\t"
)

print("#***# Demography1 #***#")
display(Demography1_columns_test_postcommit_df.head())

print("#***# PapuansOutOfAfrica_10J19 #***#")
display(PapuansOutOfAfrica_10J19_columns_test_postcommit_df.head())

#***# Demography1 #***#


,iteration,haplotype,pop,start,end,length,admix_event,admixtime,snps,Seq_DEN
0,0,0,Ingroup,126443,158290,31847,Distant_DEN>Ingroup,1551.7,7,5
1,0,0,Ingroup,206139,223580,17441,Distant_DEN>Ingroup,1551.7,8,4
2,0,0,Ingroup,1006542,1007305,763,Distant_DEN>Ingroup,1551.7,0,0
3,0,0,Ingroup,1180240,1290919,110679,Distant_DEN>Ingroup,1551.7,27,13
4,0,0,Ingroup,1342407,1344305,1898,Distant_DEN>Ingroup,1551.7,2,0


#***# PapuansOutOfAfrica_10J19 #***#


,iteration,haplotype,pop,start,end,length,admix_event,admixtime,snps,DenA,NeaA
0,0,0,Papuan,244738,253672,8934,CHB>Papuan,1656.3,1,0,0
1,0,0,Papuan,359691,459872,100181,CHB>Papuan,1656.3,32,0,0
2,0,0,Papuan,561083,572585,11502,CHB>Papuan,1656.3,2,0,0
3,0,0,Papuan,845473,918344,72871,CHB>Papuan,1409.3,19,0,0
4,0,0,Papuan,1526912,1706283,179371,CHB>Papuan,386.5,46,0,8


In [ ]:
# Sanity check the number of iterations.
# Expected output: [0, 1, 2, 3, 4]
print(
    f"Demography1 iterations:{Demography1_columns_test_postcommit_df.iteration.unique()}"
)
print(
    f"PapuansOutOfAfrica_10J19 iterations:{PapuansOutOfAfrica_10J19_columns_test_postcommit_df.iteration.unique()}"
)

Demography1 iterations:[0 1 2 3 4]
PapuansOutOfAfrica_10J19 iterations:[0 1 2 3 4]


In [ ]:
# Sanity check the segment lengths.
Demography1_columns_test_postcommit_df["end_minus_start"] = (
    Demography1_columns_test_postcommit_df["end"]
    - Demography1_columns_test_postcommit_df["start"]
)
PapuansOutOfAfrica_10J19_columns_test_postcommit_df["end_minus_start"] = (
    PapuansOutOfAfrica_10J19_columns_test_postcommit_df["end"]
    - PapuansOutOfAfrica_10J19_columns_test_postcommit_df["start"]
)


print(
    f"Demography1 does end - start = length?: {Demography1_columns_test_postcommit_df['end_minus_start'].equals(Demography1_columns_test_postcommit_df['length'])}"
)
print(
    f"PapuansOutOfAfrica_10J19 does end - start = length?: {PapuansOutOfAfrica_10J19_columns_test_postcommit_df['end_minus_start'].equals(PapuansOutOfAfrica_10J19_columns_test_postcommit_df['length'])}"
)

Demography1 does end - start = length?: True
PapuansOutOfAfrica_10J19 does end - start = length?: True
